# Racial Conclusions

Builds yes/no totals for the racial buckets using the **same `groups` mapping as `conclusions/get_conclusions.ipynb`**. Renders a stacked horizontal bar chart and saves it as both PNG and SVG. The companion notebook `racial_distributions.ipynb` renders a frequency-only bar chart from the same data.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

_IMGS = Path.cwd() / "imgs"
_IMGS.mkdir(parents=True, exist_ok=True)
_REPO = _IMGS.parents[2]  # .../viz
sys.path.insert(0, str(_REPO / "conclusions"))

from seaborn_bar_utils import render_stacked_horizontal

In [2]:
df = pd.read_csv(_REPO / "data" / "papers.csv")

paper_conclusions_lst = df["Conclusions"].tolist()
conclusions_mps: dict[str, dict[str, int]] = {}

for paper in paper_conclusions_lst:
    if isinstance(paper, float):
        continue

    for conclusion in str(paper).split(","):
        try:
            value = re.sub(r"\s*\([^)]*\)", "", conclusion).strip()
            key, val = (part.strip() for part in value.split(":", 1))
            if not key or not val:
                continue

            key_norm = key.strip()
            yn = val.strip().lower().split(None, 1)[0]

            if key_norm not in conclusions_mps:
                conclusions_mps[key_norm] = {"yes": 0, "no": 0}

            if yn == "yes":
                conclusions_mps[key_norm]["yes"] += 1
            elif yn == "no":
                conclusions_mps[key_norm]["no"] += 1
        except ValueError:
            continue

len(conclusions_mps)

204

In [3]:
groups = {
    'White': ['White', 'Caucasion', 'Whites', 'Light Skin Tone',
              'white', 'Non hispanic white', 'Caucasian'],
    'Black': ['Black', 'African American', 'Dark Skin Tone', 'non hispanic black',
              'black', 'African', 'black or african american'],
    'Hispanic': ['Hispanic', 'Hispanic or Latino', 'hispanic', 'hispanic or latino',
                 'Hispanic or latino'],
    "Asian": ['East Asians', 'Asians', 'Asian', 'Arab', 'Middle Eastern',
              'east asian', 'middle eastern', 'south asian',
              'Middle Eastern or North African', 'asian'],
    "Native American": ['Native American or Indigenous', 'Native American',
                        'American Indian or Alaska Native',
                        'American indian/native american',
                        'native american or american indian'],
    "Pacific Islander": ['Native Hawaiian or Other Pacific Islander',
                         'Pacific Islander'],
}

racial_conclusions: dict[str, dict[str, int]] = {}
for new_name, categories in groups.items():
    yes_total = sum(conclusions_mps.get(cat, {"yes": 0})["yes"] for cat in categories)
    no_total = sum(conclusions_mps.get(cat, {"no": 0})["no"] for cat in categories)
    racial_conclusions[new_name] = {"yes": yes_total, "no": no_total}

racial_conclusions

{'White': {'yes': 9, 'no': 33},
 'Black': {'yes': 28, 'no': 16},
 'Hispanic': {'yes': 11, 'no': 12},
 'Asian': {'yes': 19, 'no': 14},
 'Native American': {'yes': 7, 'no': 3},
 'Pacific Islander': {'yes': 3, 'no': 1}}

In [4]:
json_path = _IMGS / "racial_conclusions.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(racial_conclusions, f, indent=4, sort_keys=True)

png_path = _IMGS / "racial_conclusions.png"
svg_path = _IMGS / "racial_conclusions.svg"

render_stacked_horizontal(json_path, [png_path, svg_path])

print(f"Wrote {json_path}")
print(f"Wrote {png_path}")
print(f"Wrote {svg_path}")

Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/racial_conclusions/v1/racial_conclusions.json
Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/racial_conclusions/v3/racial_conclusions.png
Wrote /Users/josh/Desktop/harvard/kempner/viz/conclusions/racial_conclusions/v3/racial_conclusions.svg
